# EEG Feature Extraction - Hierarchical Strategies

This notebook extracts EEG features from preprocessed EEG data using a **principled granularity hierarchy**.

**Parameterized**: Set `STRATEGY` to choose feature extraction level.

**Run this notebook ONCE per strategy after EEG preprocessing completes.**

All other model notebooks will load the saved features instead of re-extracting.

---

## Feature Extraction Hierarchy

| Level | Strategy | Features | Description | Builds On |
|-------|----------|----------|-------------|----------|
| 0 | `channels_raw` | 20 | Total power per electrode | Base |
| 1 | `regional_raw` | 4 | Regional averages of total power | channels_raw |
| 2 | `channels_bands` | 80 | 20 channels × 4 frequency bands | channels_raw |
| 3 | `regional_bands` | 16 | 4 regions × 4 frequency bands | channels_bands |
| 4 | `extended` | 160 | channels_bands + lateralization + temporal | channels_bands |

Each level builds on the previous, allowing systematic ablation studies:
- Does band decomposition help? Compare Level 0 vs Level 2
- Does regional aggregation help? Compare Level 0 vs Level 1, or Level 2 vs Level 3
- Do derived features help? Compare Level 2 vs Level 4

In [37]:
# ============================================================================
# CONFIGURATION: Set extraction strategy and time window
# ============================================================================
STRATEGY = 'regional_bands'  # Options: 'channels_raw', 'regional_raw', 'channels_bands', 'regional_bands', 'extended'
WINDOW = 'pre'  # Options: 'pre' (display window, before decision), 'post' (review window, after decision)
# ============================================================================

import sys
sys.path.append('../..')

import pickle
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Force reload of the module to pick up any changes
import importlib
import src.features.eeg_features
importlib.reload(src.features.eeg_features)

from src.features.eeg_features import (
    extract_eeg_features,
    get_feature_metadata,
    get_strategy_hierarchy,
    CHANNEL_NAMES,
    CHANNEL_REGIONS,
    FREQ_BANDS,
    PREPROCESSED_EEG_PRE,
    PREPROCESSED_EEG_POST,
)

# Select EEG data path based on window
PREPROCESSED_EEG_PKL = PREPROCESSED_EEG_PRE if WINDOW == 'pre' else PREPROCESSED_EEG_POST
window_desc = "pre-decision (display)" if WINDOW == 'pre' else "post-decision (review)"

# Display hierarchy
hierarchy = get_strategy_hierarchy()
print(f"\n{'='*70}")
print(f"EEG FEATURE EXTRACTION: {STRATEGY.upper()} (Level {hierarchy[STRATEGY]['level']})")
print(f"Time window: {window_desc}")
print(f"{'='*70}\n")
print(f"Feature extraction started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nStrategy Hierarchy:")
for strat, info in hierarchy.items():
    marker = "-->" if strat == STRATEGY else "   "
    print(f"  {marker} Level {info['level']}: {strat:20s} ({info['n_features']:3d} features)")


EEG FEATURE EXTRACTION: REGIONAL_BANDS (Level 3)
Time window: pre-decision (display)

Feature extraction started: 2026-04-17 10:00:52

Strategy Hierarchy:
      Level 0: channels_raw         ( 20 features)
      Level 1: regional_raw         (  4 features)
      Level 2: channels_bands       ( 80 features)
  --> Level 3: regional_bands       ( 16 features)
      Level 4: extended             (112 features)


## 1. Load Preprocessed EEG Data

Load the preprocessed EEG pickle file based on selected time window:
- **Pre-decision (display window)**: `display_window_2s.pkl` - EEG from stimulus onset to decision (−2s to 0s relative to decision)
- **Post-decision (review window)**: `review_window_2s.pkl` - EEG from decision to feedback (0s to +2s relative to decision)

In [38]:
eeg_data_path = PREPROCESSED_EEG_PKL

print(f"Loading EEG data from: {eeg_data_path}")
with open(eeg_data_path, 'rb') as f:
    eeg_df = pickle.load(f)

print(f"\n✓ Loaded {len(eeg_df)} trials")
print(f"  Unique subjects: {eeg_df['subject_date_id'].nunique()}")

# Check EEG data structure (column is 'eeg' in new files)
sample_eeg = eeg_df['eeg'].iloc[0]
print(f"  EEG array shape: {sample_eeg.shape} (channels × time)")
print(f"\nColumns: {eeg_df.columns.tolist()}")

Loading EEG data from: /Users/pranmodu/Projects/columbia/liinc_eye/data/eeg/display_window_2s.pkl

✓ Loaded 10212 trials
  Unique subjects: 80
  EEG array shape: (20, 512) (channels × time)

Columns: ['subject_date_id', 'trial_id', 'eeg', 'n_valid_pre', 'ambiguity', 'invest', 'condition', 'subject_id']


## 2. Channel and Region Configuration

Display the channel configuration used for feature extraction.

In [39]:
print("\n" + "="*70)
print("CHANNEL CONFIGURATION (from chan_locs.sfp)")
print("="*70)

print(f"\n20 EEG Channels (10-20 system):")
print(f"  {', '.join(CHANNEL_NAMES)}")

print(f"\n4 Brain Regions:")
for region, channels in CHANNEL_REGIONS.items():
    print(f"  {region:10s}: {', '.join(channels)}")

print(f"\n4 Frequency Bands:")
for band, (f_low, f_high) in FREQ_BANDS.items():
    print(f"  {band:6s}: {f_low:4.1f} - {f_high:4.1f} Hz")


CHANNEL CONFIGURATION (from chan_locs.sfp)

20 EEG Channels (10-20 system):
  Fp1, F7, F8, T4, T6, T5, T3, Fp2, O1, P3, Pz, F3, Fz, F4, C4, P4, POz, C3, Cz, O2

4 Brain Regions:
  Frontal   : Fp1, Fp2, F7, F3, Fz, F4, F8
  Central   : T3, C3, Cz, C4, T4
  Parietal  : T5, P3, Pz, P4, T6
  Occipital : O1, POz, O2

4 Frequency Bands:
  Delta :  0.5 -  4.0 Hz
  Theta :  4.0 -  8.0 Hz
  Alpha :  8.0 - 13.0 Hz
  Beta  : 13.0 - 30.0 Hz


## 3. Extract EEG Features

Extract features based on selected strategy:

### Level 0: channels_raw (20 features)
- Total (broadband) power per electrode
- **Features:** `eeg_{channel}` for each of 20 channels
- **Use case:** Simplest baseline, tests if spatial pattern alone carries signal

### Level 1: regional_raw (4 features)
- Regional averages of total power
- **Features:** `eeg_{Frontal,Central,Parietal,Occipital}`
- **Use case:** Most compact representation, tests regional differences

### Level 2: channels_bands (80 features)
- Band power per channel (20 channels × 4 bands)
- **Features:** `eeg_{band}_{channel}`
- **Use case:** Full spatial + spectral resolution

### Level 3: regional_bands (16 features)
- Regional average band power (4 regions × 4 bands)
- **Features:** `eeg_{band}_{region}`
- **Use case:** Standard approach, balances detail with interpretability

### Level 4: extended (160 features)
- channels_bands + temporal dynamics + lateralization
- **Channel-band power:** 80 features
- **Temporal dynamics:** `eeg_{band}_{region}_{mean,std,slope}` (48 features)
- **Lateralization:** `eeg_{band}_{pair}_lateralization` (32 features)
- **Use case:** Maximum feature richness for best performance

In [40]:
print(f"\nExtracting EEG features using '{STRATEGY}' strategy...\n")

eeg_features_df = extract_eeg_features(
    eeg_df=eeg_df,
    strategy=STRATEGY,
    fs=256,
    verbose=True
)

# Get feature columns
eeg_cols = [c for c in eeg_features_df.columns if c.startswith('eeg_')]

print(f"\n{'='*70}")
print(f"✓ Extracted {len(eeg_cols)} EEG features")
print(f"  Example features: {eeg_cols[:5]}")
if len(eeg_cols) > 5:
    print(f"  ... and {len(eeg_cols) - 5} more")
print(f"{'='*70}")


Extracting EEG features using 'regional_bands' strategy...

Extracting EEG features using 'regional_bands' strategy (Level 3)...
  EEG column: eeg
  Sampling rate: 256 Hz
  Trials: 10212
✓ Extracted 16 EEG features
  Regional-band power: 16 features (4 regions × 4 bands)

✓ Extracted 16 EEG features
  Example features: ['eeg_Delta_Frontal', 'eeg_Delta_Central', 'eeg_Delta_Parietal', 'eeg_Delta_Occipital', 'eeg_Theta_Frontal']
  ... and 11 more


## 4. Inspect Features

In [41]:
# Display sample data
print("\nSample EEG features:")
display(eeg_features_df.head(3))

# Feature statistics
print("\nFeature statistics:")
display(eeg_features_df[eeg_cols].describe())


Sample EEG features:


,subject_id,trial_id,eeg_Delta_Frontal,eeg_Delta_Central,eeg_Delta_Parietal,eeg_Delta_Occipital,eeg_Theta_Frontal,eeg_Theta_Central,eeg_Theta_Parietal,eeg_Theta_Occipital,eeg_Alpha_Frontal,eeg_Alpha_Central,eeg_Alpha_Parietal,eeg_Alpha_Occipital,eeg_Beta_Frontal,eeg_Beta_Central,eeg_Beta_Parietal,eeg_Beta_Occipital
0,0731_1000_U9TEJGM,4_0731_1000_U9TEJGM,0.003893,0.002474,0.002135,0.003595,0.001237,0.001027,0.000759,0.001253,0.000324,0.000328,0.000226,0.000465,0.000634,0.000689,0.000585,0.000474
1,0731_1000_U9TEJGM,5_0731_1000_U9TEJGM,0.002309,0.001396,0.001220,0.001842,0.000485,0.000697,0.000568,0.001315,0.000225,0.000362,0.000232,0.000592,0.000422,0.000483,0.000440,0.000452
2,0731_1000_U9TEJGM,6_0731_1000_U9TEJGM,0.003735,0.003141,0.002787,0.004734,0.001489,0.001257,0.000831,0.001430,0.000324,0.000346,0.000242,0.000453,0.000423,0.000530,0.000522,0.000441



Feature statistics:


,eeg_Delta_Frontal,eeg_Delta_Central,eeg_Delta_Parietal,eeg_Delta_Occipital,eeg_Theta_Frontal,eeg_Theta_Central,eeg_Theta_Parietal,eeg_Theta_Occipital,eeg_Alpha_Frontal,eeg_Alpha_Central,eeg_Alpha_Parietal,eeg_Alpha_Occipital,eeg_Beta_Frontal,eeg_Beta_Central,eeg_Beta_Parietal,eeg_Beta_Occipital
count,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000,6020.000000
mean,0.110184,0.099400,0.105433,0.117160,0.050981,0.046612,0.050333,0.060076,0.035806,0.036680,0.039414,0.041678,0.049018,0.052952,0.048090,0.044692
std,0.379935,0.280734,0.310102,0.461335,0.090465,0.077531,0.077817,0.088677,0.061670,0.069711,0.063750,0.065471,0.114962,0.181560,0.087821,0.074799
min,0.000064,0.000079,0.000056,0.000058,0.000057,0.000043,0.000034,0.000038,0.000033,0.000025,0.000021,0.000015,0.000036,0.000027,0.000025,0.000022
25%,0.003830,0.002955,0.002188,0.002190,0.001114,0.000880,0.000700,0.000853,0.000449,0.000403,0.000354,0.000433,0.000762,0.000641,0.000548,0.000498
50%,0.027527,0.023236,0.019909,0.017181,0.010223,0.008892,0.008690,0.009463,0.006600,0.005671,0.005470,0.006474,0.012707,0.013924,0.010712,0.008727
75%,0.170870,0.158869,0.179544,0.189406,0.078996,0.076647,0.086699,0.097796,0.054974,0.050098,0.061268,0.064963,0.068431,0.071010,0.076399,0.073386
max,24.256651,17.473586,20.708931,33.422238,2.883570,1.897233,1.681634,1.266425,1.301293,2.058726,1.211982,0.896225,6.059796,10.239835,2.236021,2.377926


## 5. Categorize Features by Type

Group features by their type based on the extraction strategy.

In [42]:
feature_categories = {}

if STRATEGY == 'channels_raw':
    feature_categories['channel_power'] = eeg_cols
    print(f"\nChannel power features: {len(feature_categories['channel_power'])}")
    print(f"  Channels: {[c.replace('eeg_', '') for c in eeg_cols]}")

elif STRATEGY == 'regional_raw':
    feature_categories['regional_power'] = eeg_cols
    print(f"\nRegional power features: {len(feature_categories['regional_power'])}")
    print(f"  Regions: {[c.replace('eeg_', '') for c in eeg_cols]}")

elif STRATEGY == 'channels_bands':
    for band in FREQ_BANDS.keys():
        band_cols = [c for c in eeg_cols if f'eeg_{band}_' in c]
        feature_categories[f'{band}_channels'] = band_cols
    print(f"\nChannel-band power features by band:")
    for band, cols in feature_categories.items():
        print(f"  {band}: {len(cols)} features")

elif STRATEGY == 'regional_bands':
    for band in FREQ_BANDS.keys():
        band_cols = [c for c in eeg_cols if f'eeg_{band}_' in c]
        feature_categories[f'{band}_regional'] = band_cols
    print(f"\nRegional-band power features by band:")
    for band, cols in feature_categories.items():
        print(f"  {band}: {len(cols)} features")

elif STRATEGY == 'extended':
    # Separate by feature type
    channel_band_cols = [c for c in eeg_cols if not any(x in c for x in 
                        ['_mean', '_std', '_slope', '_lateralization'])]
    temporal_cols = [c for c in eeg_cols if any(x in c for x in 
                    ['_mean', '_std', '_slope'])]
    lat_cols = [c for c in eeg_cols if '_lateralization' in c]
    
    feature_categories['channel_band_power'] = channel_band_cols
    feature_categories['temporal_dynamics'] = temporal_cols
    feature_categories['lateralization'] = lat_cols
    
    print(f"\nExtended features by type:")
    print(f"  Channel-band power: {len(channel_band_cols)} features")
    print(f"  Temporal dynamics: {len(temporal_cols)} features")
    print(f"  Lateralization: {len(lat_cols)} features")

print(f"\nTotal features: {len(eeg_cols)}")


Regional-band power features by band:
  Delta_regional: 4 features
  Theta_regional: 4 features
  Alpha_regional: 4 features
  Beta_regional: 4 features

Total features: 16


## 6. Prepare Metadata

Create comprehensive metadata for reproducibility.

In [43]:
metadata = get_feature_metadata(STRATEGY)
metadata.update({
    'extraction_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_trials': len(eeg_features_df),
    'n_subjects': eeg_features_df['subject_id'].nunique(),
    'input_file': eeg_data_path,
    'description': f'EEG features extracted using {STRATEGY} strategy (Level {metadata["level"]})'
})

print("\nMetadata:")
for key, value in metadata.items():
    if not isinstance(value, (dict, list)):
        print(f"  {key}: {value}")


Metadata:
  strategy: regional_bands
  level: 3
  builds_on: channels_bands
  sampling_rate: 256
  n_channels: 20
  n_regions: 4
  n_features: 16
  extraction_date: 2026-04-17 10:00:58
  n_trials: 10212
  n_subjects: 80
  input_file: /Users/pranmodu/Projects/columbia/liinc_eye/data/eeg/display_window_2s.pkl
  description: EEG features extracted using regional_bands strategy (Level 3)


## 7. Save to Pickle File

Save features with metadata for use in other notebooks.

In [44]:
# Output path based on strategy
output_dir = Path('../../data/features')
output_dir.mkdir(parents=True, exist_ok=True)

# Strategy-to-filename mapping
strategy_filenames = {
    'channels_raw': 'eeg_features_channels_raw.pkl',
    'regional_raw': 'eeg_features_regional_raw.pkl',
    'channels_bands': 'eeg_features_channels_bands.pkl',
    'regional_bands': 'eeg_features_regional_bands.pkl',
    'extended': 'eeg_features_extended.pkl'
}

output_filename = strategy_filenames[STRATEGY]
output_path = output_dir / output_filename

# Prepare output data (matching main feature extraction format)
output_data = {
    'eeg_features_df': eeg_features_df,
    'feature_columns': eeg_cols,
    'feature_categories': feature_categories,
    'metadata': metadata
}

# Save
with open(output_path, 'wb') as f:
    pickle.dump(output_data, f)

file_size_mb = output_path.stat().st_size / 1024 / 1024

print(f"\n{'='*70}")
print(f"✓ EEG features saved to: {output_path}")
print(f"  File size: {file_size_mb:.2f} MB")
print(f"  Trials: {len(eeg_features_df)}")
print(f"  Subjects: {eeg_features_df['subject_id'].nunique()}")
print(f"  Features: {len(eeg_cols)}")
print(f"\nCompleted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}")


✓ EEG features saved to: ../../data/features/eeg_features_regional_bands.pkl
  File size: 1.50 MB
  Trials: 10212
  Subjects: 80
  Features: 16

Completed: 2026-04-17 10:00:58


## 8. Verification

Verify the saved file can be loaded correctly.

In [45]:
# Test loading
print("\nVerifying saved file...")
with open(output_path, 'rb') as f:
    test_data = pickle.load(f)

print(f"✓ File loads successfully")
print(f"  Keys: {list(test_data.keys())}")
print(f"  Features: {len(test_data['feature_columns'])}")
print(f"  Trials: {len(test_data['eeg_features_df'])}")
print(f"  Strategy: {test_data['metadata']['strategy']}")
print(f"  Level: {test_data['metadata']['level']}")
print("\n✓ Verification complete!")


Verifying saved file...
✓ File loads successfully
  Keys: ['eeg_features_df', 'feature_columns', 'feature_categories', 'metadata']
  Features: 16
  Trials: 10212
  Strategy: regional_bands
  Level: 3

✓ Verification complete!


---

## Usage in Other Notebooks

To use these EEG features in fusion models or other analyses:

```python
import pickle

# Load EEG features (choose the strategy you want)
strategy = 'channels_raw'  # or 'regional_raw', 'channels_bands', 'regional_bands', 'extended'
with open(f'../../data/features/eeg_features_{strategy}.pkl', 'rb') as f:
    eeg_data = pickle.load(f)

eeg_features_df = eeg_data['eeg_features_df']
eeg_cols = eeg_data['feature_columns']
metadata = eeg_data['metadata']

print(f"Loaded {metadata['strategy']} (Level {metadata['level']}): {len(eeg_cols)} features")

# Merge with other modalities
merged_df = merged_df.merge(
    eeg_features_df,
    on=['subject_id', 'trial_id'],
    how='inner'
)
```

---

## Strategy Hierarchy Reference

| Level | Strategy | Features | Builds On | Output File |
|-------|----------|----------|-----------|-------------|
| 0 | `channels_raw` | 20 | Base | `eeg_features_channels_raw.pkl` |
| 1 | `regional_raw` | 4 | channels_raw | `eeg_features_regional_raw.pkl` |
| 2 | `channels_bands` | 80 | channels_raw | `eeg_features_channels_bands.pkl` |
| 3 | `regional_bands` | 16 | channels_bands | `eeg_features_regional_bands.pkl` |
| 4 | `extended` | 160 | channels_bands | `eeg_features_extended.pkl` |

---

## Channel Configuration

All features use standardized channel configuration from `data/eeg/chan_locs.sfp`:

**20 EEG Channels (10-20 system):**
```
Fp1, F7, F8, T4, T6, T5, T3, Fp2, O1, P3, Pz, F3, Fz, F4, C4, P4, POz, C3, Cz, O2
```

**4 Brain Regions:**
- **Frontal:** Fp1, Fp2, F7, F3, Fz, F4, F8
- **Central:** T3, C3, Cz, C4, T4
- **Parietal:** T5, P3, Pz, P4, T6
- **Occipital:** O1, POz, O2

**4 Frequency Bands:**
- Delta: 0.5-4 Hz
- Theta: 4-8 Hz
- Alpha: 8-13 Hz
- Beta: 13-30 Hz